# Fine-tune Laya untuk gerbang PANDU

Notebook ini melatih `laya-multilingual` dengan dataset hasil `bangun_dataset.py`, lalu
**mengukurnya di sini sebelum dipasang di server** (`api/docs/kaggle.md`): set kalibrasi
dulu, lalu set uji sekali, keduanya dengan ambang produksi. Checkpoint hanya dipasang
bila tabel keputusan di akhir notebook menyatakan LULUS.

**Sebelum Run All**
1. Unggah `kaggle.zip` dari `bangun_dataset.py` sebagai Kaggle Dataset privat, atau sebagai
   versi baru dataset lama. Untuk putaran 4: `api/eval/laya/data/r4/kaggle.zip`. Isinya
   `latih_laya.py`, `evaluasi.py`, train, val, uji, kalibrasi, dan manifest.
2. Panel kanan:
   - *Add Input* → dataset tadi (versi terbaru)
   - *Accelerator* → **GPU T4 x1** (atau P100)
   - *Internet* → **On** (untuk `pip install` dan unduhan model)
3. *Save Version* → *Save & Run All*. Perkiraan: 10–20 menit di satu T4.

Hasil di tab *Output*:
- `laya-pandu-gerbang.tgz` (±600 MB)
- `hasil_kalibrasi.jsonl`
- `hasil_uji.jsonl`
- `keputusan.json`

In [ ]:
!nvidia-smi -L
!pip install -q "laya @ git+https://github.com/NandhaKishorM/laya@v0.3.22"
import laya
import torch
import transformers

print(
    "laya",
    laya.__version__ if hasattr(laya, "__version__") else "?",
    "| torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| cuda",
    torch.cuda.is_available(),
)

In [ ]:
# Cari folder input yang berisi train.jsonl dan skrip, lalu salin ke /kaggle/working.
import glob
import os
import shutil
from pathlib import Path

train = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert train, (
    "train.jsonl tidak ditemukan: tambahkan Kaggle Dataset berisi file dari api/eval/laya/"
)
DATA = os.path.dirname(train[0])
KERJA = "/kaggle/working"
for nama in ("latih_laya.py", "evaluasi.py"):
    sumber = glob.glob(f"/kaggle/input/**/{nama}", recursive=True)
    assert sumber, f"{nama} tidak ditemukan di input"
    shutil.copy(sumber[0], KERJA)
os.makedirs(f"{KERJA}/data", exist_ok=True)
for nama in ("train.jsonl", "val.jsonl", "uji.jsonl", "kalibrasi.jsonl", "manifest.json"):
    assert os.path.exists(os.path.join(DATA, nama)), f"{nama} tidak ada di dataset"
    shutil.copy(os.path.join(DATA, nama), f"{KERJA}/data/")
print(Path(f"{KERJA}/data/manifest.json").read_text()[:1500])

## Latih

Loss dan hiperparameter mengikuti skrip resmi Laya (RLCD). Bobot terbaik dipilih
menurut cross-entropy validasi, dan temperature dikalibrasi pada set validasi.
Metrik per epoch: akurasi, dan jumlah pesan akademik yang akan terblokir menurut
ambang bawaan `latih_laya.py`. Ambang produksi baru dipakai di bagian *Ukur*.

In [ ]:
%cd /kaggle/working
!python latih_laya.py --data data --out laya-pandu-gerbang --epochs 4 --micro-batch 16

## Ukur sebelum dipasang

Ambangnya sama persis dengan produksi (`gate.GatePolicy`) dan dengan baseline r1
(`docs/kaggle.md` §2), jadi angkanya bisa dibandingkan langsung:
- `--ambang 0.7`: smalltalk dan malicious (`LAYA_BLOCK_THRESHOLD`).
- `--ambang-luar 0.7`: out_of_scope (`LAYA_OUT_OF_SCOPE_THRESHOLD`).
- `--ambang-nonsense 0.95`: nonsense (`LAYA_NONSENSE_THRESHOLD`).

Putaran 3 masih memakai 0,95 untuk smalltalk, sehingga "ipk gw skrg" (smalltalk 0,73)
tidak terhitung terblokir padahal di produksi terblokir. Sejak putaran 4 ketiga ambang
dipisah. Keputusan blokir dibaca dari `answer_confidence`, seperti `gate.py`. Urutannya:
set kalibrasi dulu, lalu set uji **sekali**. Jangan mengulang set uji untuk memilih
ambang atau epoch.

In [ ]:
OUT = "laya-pandu-gerbang"
AMBANG, AMBANG_LUAR, AMBANG_NONSENSE = 0.7, 0.7, 0.95
UKUR = (
    f"--device cuda --ambang {AMBANG} --ambang-luar {AMBANG_LUAR} "
    f"--ambang-nonsense {AMBANG_NONSENSE}"
)
!python evaluasi.py --uji data/kalibrasi.jsonl --model $OUT $UKUR --out hasil_kalibrasi.jsonl

In [ ]:
!python evaluasi.py --uji data/uji.jsonl --model $OUT $UKUR --out hasil_uji.jsonl

## Keputusan: bandingkan dengan r1

Baseline r1 (checkpoint yang sedang dipakai produksi) diukur 2026-11-20 lewat server dengan
ambang yang sama (`docs/kaggle.md` §2). Angka syarat di sel berikut berlaku untuk dataset
putaran 4 (uji 145, kalibrasi 83). Kalau dataset berubah, perbarui angkanya.

Baris *info* tidak menentukan keputusan. Isinya kelemahan r3 yang ditambal di putaran 4:
permintaan data mahasiswa lain (malicious) dan uang di luar kampus dengan unit Keuangan.

- **LULUS**: unduh hasil, lalu pasang (`docs/kaggle.md` §9).
- **TIDAK LULUS**: jangan pasang. Tinjau pesan yang terblokir di bawah, perbaiki datanya,
  lalu latih lagi dengan versi dataset baru.

In [ ]:
import json


def baca(p):
    return [json.loads(b) for b in open(p, encoding="utf-8") if b.strip()]


uji, kal = baca("hasil_uji.jsonl"), baca("hasil_kalibrasi.jsonl")


def blok(rows, harap, *asal):
    g = [r for r in rows if r["expected"] == harap and (not asal or r["asal"] in asal)]
    return sum(r["blok_answer"] for r in g), len(g)


def akurasi(rows, asal):
    g = [r for r in rows if r["asal"] == asal]
    return sum(r["choice"] == r["expected"] for r in g), len(g)


def jumlah(*pasang):
    return sum(v for v, _ in pasang), sum(n for _, n in pasang)


def nol(v):
    return v == 0


# (ukuran, (nilai, n), syarat, baseline r1); syarat None = info saja
SYARAT = [
    ("akademik terblokir, uji semua", blok(uji, "academic"), nol, "4/89"),
    ("akademik terblokir, uji asal r3", blok(uji, "academic", "r3"), nol, "4/19"),
    ("akademik terblokir, uji asal r4", blok(uji, "academic", "r4"), nol, "0/12"),
    ("akademik terblokir, kalibrasi", blok(kal, "academic"), nol, "0/41"),
    ("luar topik terblokir, uji semua", blok(uji, "out_of_scope"), lambda v: v >= 24, "24/30"),
    ("luar topik terblokir, kalibrasi", blok(kal, "out_of_scope"), lambda v: v >= 18, "18/24"),
    ("akurasi uji asal r1", akurasi(uji, "r1"), lambda v: v >= 55, "57/64"),
    (
        "info: malicious terblokir, r4",
        jumlah(blok(uji, "malicious", "r4"), blok(kal, "malicious", "kalibrasi-r4")),
        None,
        "0/5",
    ),
    (
        "info: luar topik terblokir, r3+r4",
        jumlah(
            blok(uji, "out_of_scope", "r3", "r4"),
            blok(kal, "out_of_scope", "kalibrasi-r3", "kalibrasi-r4"),
        ),
        None,
        "12/16",
    ),
]
lulus = True
print(f"{'ukuran':36s} {'baru':>7s} {'r1':>7s}  hasil")
for nama, (v, n), syarat, r1 in SYARAT:
    ok = syarat is None or syarat(v)
    lulus = lulus and ok
    hasil = "-" if syarat is None else ("ok" if ok else "GAGAL")
    print(f"{nama:36s} {f'{v}/{n}':>7s} {r1:>7s}  {hasil}")
print()
print("KEPUTUSAN:", "LULUS, boleh dipasang" if lulus else "TIDAK LULUS, jangan dipasang")
for r in uji + kal:
    if r["expected"] == "academic" and r["blok_answer"]:
        p = r["answer_confidence"]
        print(f"  akademik terblokir: {r['asal']} {r['choice']} p={p:.2f} {r['pesan'][:60]}")
with open("keputusan.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "lulus": lulus,
            "ambang": AMBANG,
            "ambang_luar": AMBANG_LUAR,
            "ambang_nonsense": AMBANG_NONSENSE,
            "ukuran": {
                nama: {"baru": f"{v}/{n}", "r1": r1, "syarat": syarat is not None}
                for nama, (v, n), syarat, r1 in SYARAT
            },
        },
        f,
        ensure_ascii=False,
        indent=1,
    )

## Pembanding: checkpoint dasar (zero-shot)

Hanya untuk konteks, tidak memengaruhi keputusan.

In [ ]:
!python evaluasi.py --uji data/uji.jsonl --model basis $UKUR | head -20

In [ ]:
!cat laya-pandu-gerbang/laporan_latih.json | head -80
!tar czf laya-pandu-gerbang.tgz laya-pandu-gerbang && ls -la laya-pandu-gerbang.tgz